# TAB 67a: exp67a_dynamic_condition (EXP-67a: Dynamic Condition Token + Physical Raw L1 + Anti-Hijack Guard) - Colab
# Nền tảng cốt lõi EXP-67:
# 1. Phục hồi Raw L1 vật lý (--pred_loss_type raw_l1): Tối ưu hóa trực tiếp trong không gian đặc trưng biên độ thực, loại bỏ ép cầu Cosine.
# 2. Chặn Gradient Hijack (--anti_hijack_ratio 0.25): Khóa trần các hàm phụ, bảo đảm JEPA prediction loss luôn chiếm >= 80% gradient suốt quá trình train.
# 3. Kế thừa Contiguous Wedge Masking (--contiguous_masking true) và Self-Focus Weighting (--self_focus_gamma 0.5).
# 4. HƯỚNG 1 ĐỘC LẬP (EXP-67a): Triệt tiêu bảng tra tĩnh 16 file (--dynamic_condition_token true), lấy mẫu sound-metal reference patch động theo từng scan.
# Giữ nguyên Two-Stream Backbone (Multiplicative FiLM) để cô lập duy nhất biến số Condition Token.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os, glob
DATA = '/content/drive/MyDrive/PECT_Dataset/'
DRIVE_RUNS = '/content/drive/MyDrive/PECT_Dataset/runs/'
os.makedirs(DRIVE_RUNS, exist_ok=True)
# Verify: TDMS + GT masks phai thay tu Drive
tdms = [f for f in glob.glob(os.path.join(DATA, '**', '*.tdms'), recursive=True) if not f.endswith('_index.tdms')]
print('TDMS:', len(tdms))
for p in [os.path.join(DATA, 'ground_truth/corrosion/corrosion_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v1/rivet_v1_gt_mask.npy'),
          os.path.join(DATA, 'ground_truth/rivet_v2/rivet_v2_gt_mask.npy')]:
    print(('OK  ' if os.path.exists(p) else 'MISS') + ' ' + p)

TDMS: 81
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/corrosion/corrosion_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v1/rivet_v1_gt_mask.npy
OK   /content/drive/MyDrive/PECT_Dataset/ground_truth/rivet_v2/rivet_v2_gt_mask.npy


In [3]:
!test -d GraduationProject && (cd GraduationProject && git pull --ff-only) || git clone https://github.com/TranAnh35/GraduationProject.git
!cd GraduationProject && git switch Define_PECT_JEPA && git pull --ff-only && git log --oneline -3

Cloning into 'GraduationProject'...
remote: Enumerating objects: 2566, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 2566 (delta 36), reused 41 (delta 26), pack-reused 2498 (from 1)
Receiving objects: 100% (2566/2566), 40.38 MiB | 25.79 MiB/s, done.
Resolving deltas: 100% (1718/1718), done.
branch 'Define_PECT_JEPA' set up to track 'origin/Define_PECT_JEPA'.
Switched to a new branch 'Define_PECT_JEPA'
Already up to date.
376db6a (HEAD -> Define_PECT_JEPA, origin/Define_PECT_JEPA) feat(ablation): restructure EXP-66 ablation into 66b (masking), 66c (loss), and 66d (self-focus mechanism)
bc4ec92 feat(colab): add dedicated parallel training notebooks for EXP-66b, EXP-67, and EXP-68
c288fd2 feat(ablation): implement harder wedge masking (5 rays/60% mask), log-energy magnitude coupling and unblock divergence guard


In [4]:
# [FIX] sklearn version co sample_weight trong MLPClassifier.fit -> ep dung 1.7.2 (giong env lab)
!pip install -q -U "scikit-learn==1.7.2" nptdms scipy tqdm 2>&1 | tail -2
!cd GraduationProject && python -c "import torch, sklearn, scipy, nptdms; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| sklearn:', sklearn.__version__)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.7 MB/s eta 0:00:00
torch 2.11.0+cu130 | cuda: True | sklearn: 1.7.2


In [5]:
!mkdir -p "$DRIVE_RUNS"
!cd GraduationProject && mkdir -p experiments/cross_condition && if [ ! -L experiments/cross_condition/runs ]; then mv experiments/cross_condition/runs/* "$DRIVE_RUNS/" 2>/dev/null; rm -rf experiments/cross_condition/runs && ln -s "$DRIVE_RUNS" experiments/cross_condition/runs && echo LINKED; else echo ALREADY_LINKED; fi
!cd GraduationProject && ls -la experiments/cross_condition/ | grep runs

LINKED
lrwxrwxrwx 1 root root   41 Oct 21 00:53 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [6]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# ===== EXP-67a CONFIG: Dynamic Condition Token + Raw L1 + Anti-Hijack =====
EXP_NAME = 'exp67a_dynamic_condition'

BASE_FLAGS = '--star_radii 2 4 7 --tokenizer_type energy_adaptive --latent_disentanglement true --carrier_flat_weight 0.0 --scattering_var_weight 0.5 --inv_start_epoch 11 --inv_ramp_epochs 20 --xcov_start_epoch 11 --xcov_ramp_epochs 20 --inv_cosine true --zc_cond_weight 0.1 --var_gamma 0.5 --offset_weight 0.0 --pool_var_weight 1.0 --diverge_patience 0 --device cuda'
EXTRA = '--continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --two_stream_backbone true --carrier_coupling_mode multiplicative --grounded_condition_pred true --use_condition_token true --relative_condition_operator true --condition_modulation pure_film --pred_loss_type raw_l1 --anti_hijack_ratio 0.25 --self_focus_gamma 0.5 --dynamic_condition_token true --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05'

os.environ.update({'EXP_NAME': EXP_NAME, 'BASE_FLAGS': BASE_FLAGS, 'EXTRA': EXTRA, 'DATA': DATA, 'DRIVE_RUNS': DRIVE_RUNS})
print('EXPERIMENT:', EXP_NAME)
print('FLAGS:', EXTRA)


EXPERIMENT: exp66d_self_focus
FLAGS: --continuous_pos_embed true --contiguous_masking true --predictor_scattering_only false --scan_centered_inv true --inv_predictor true --relative_energy_anchor true --bi_directional_relative_energy true --patch_polarity_align true --two_stream_backbone true --carrier_coupling_mode multiplicative --grounded_condition_pred true --use_condition_token true --relative_condition_operator true --condition_modulation pure_film --scattering_cov_weight 0.2 --inv_weight 0.2 --xcov_weight 0.05 --self_focus_gamma 0.5


In [7]:
# Symlink runs sang Drive: checkpoint va metrics ghi truc tiep len Google Drive
!mkdir -p "$DRIVE_RUNS"
!if [ -e GraduationProject/experiments/cross_condition/runs ] && [ ! -L GraduationProject/experiments/cross_condition/runs ]; then mv GraduationProject/experiments/cross_condition/runs GraduationProject/experiments/cross_condition/runs_local_bak; fi
!ln -sfn "$DRIVE_RUNS" GraduationProject/experiments/cross_condition/runs
!ls -la GraduationProject/experiments/cross_condition/ | grep runs


lrwxrwxrwx 1 root root   41 Oct 21 00:53 runs -> /content/drive/MyDrive/PECT_Dataset/runs/


In [ ]:
# Cell Train (Fresh Run): chay de huan luyen exp63 (H15)
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name $EXP_NAME --add_timestamp true $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1

[18:25:25 | INFO] Initialized Unified Experiment Directory: experiments/cross_condition/runs/exp66d_self_focus_20261020_182525
[18:25:25 | INFO] Configuration: CrossConditionConfig(exp_name='exp66d_self_focus', add_timestamp=True, base_exp_dir='experiments/cross_condition/runs', embed_dim=64, encoder_depth=4, encoder_heads=4, predictor_depth=2, predictor_heads=4, mlp_ratio=4.0, dropout=0.0, spatial_topology='concentric_star', star_radii=(2, 4, 7), grid_size=5, temporal_samples=128, num_harmonics=16, num_target_probes=9, num_context_probes=16, num_target_rays=3, continuous_pos_embed=True, contiguous_masking=True, predictor_scattering_only=False, scan_centered_inv=True, patch_energy_norm=False, median_patch_energy=False, clean_phasor_token=False, subspace_norm=False, relative_energy_anchor=True, patch_polarity_align=True, bi_directional_relative_energy=True, diffusion_attention=False, two_stream_backbone=True, carrier_coupling_mode='multiplicative', grounded_condition_pred=True, use_cond

In [8]:
# ==============================================================================
# Resume training tiep tuc tu DRIVE_RUNS (CHỈ chạy khi session trước đã dừng hẳn)
# ==============================================================================
import os, glob

# 1. Tim run folder trong DRIVE_RUNS (uu tien run moi nhat hoac chi dinh run cu the)
SPECIFIC_RUN = ""  # De trong de tu dong lay run moi nhat, hoac dien vi du: "exp60_contpos_asyminv_fullctx_20261017_223000"

if SPECIFIC_RUN:
    run_dir = os.path.join(DRIVE_RUNS, SPECIFIC_RUN)
else:
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")), key=os.path.getmtime)
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

print(f"[*] Thu muc run tren Drive: {run_dir}")

# 2. Xac dinh checkpoint (uu tien latest de tiep tuc epoch & optimizer/scaler state, fallback sang best)
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")
ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")

if os.path.exists(ckpt_latest):
    resume_ckpt = ckpt_latest
elif os.path.exists(ckpt_best):
    resume_ckpt = ckpt_best
else:
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Checkpoint resume: {resume_ckpt} ({os.path.getsize(resume_ckpt) / 1024**2:.1f} MB)")

# 3. Kiem tra tien do da train tu metrics_epoch.csv
csv_path = os.path.join(run_dir, "evaluation_results", "metrics_epoch.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(run_dir, "metrics_epoch.csv")
if os.path.exists(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        lines = [line.strip() for line in f if line.strip()]
        if len(lines) > 1:
            last_line = lines[-1].split(",")
            print(f"[*] Da hoan thanh epoch: {last_line[0]} | Val AUC gan nhat: {last_line[14] if len(last_line) > 14 else 'N/A'}")

# 4. Dam bao symlink GraduationProject/experiments/cross_condition/runs -> DRIVE_RUNS
os.system(f'mkdir -p GraduationProject/experiments/cross_condition && ln -sfn "{DRIVE_RUNS}" GraduationProject/experiments/cross_condition/runs')

# 5. Export checkpoint path sang environ va thuc hien resume training
os.environ["RESUME_CKPT"] = resume_ckpt
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.train --exp_name "$EXP_NAME" $BASE_FLAGS $EXTRA --data_dir "$DATA" --epochs 100 --eval_val_every 1 --resume_checkpoint "$RESUME_CKPT"

[*] Thu muc run tren Drive: /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525
[*] Checkpoint resume: /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/checkpoints/latest_cross_condition_jepa.pt (4.2 MB)
[*] Da hoan thanh epoch: 26 | Val AUC gan nhat: 0.8465
[00:54:03 | INFO] Initialized Unified Experiment Directory: ../drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525
[00:54:04 | INFO] Configuration: CrossConditionConfig(exp_name='exp66d_self_focus', add_timestamp=True, base_exp_dir='experiments/cross_condition/runs', embed_dim=64, encoder_depth=4, encoder_heads=4, predictor_depth=2, predictor_heads=4, mlp_ratio=4.0, dropout=0.0, spatial_topology='concentric_star', star_radii=(2, 4, 7), grid_size=5, temporal_samples=128, num_harmonics=16, num_target_probes=9, num_context_probes=16, num_target_rays=3, continuous_pos_embed=True, contiguous_masking=True, predictor_scattering_only=False, scan_centered_inv=True, patch_energy_

In [9]:
# ==============================================================================
# 1. Huấn luyện các mô hình Downstream (Linear/MLP Probes, Sizing, UDA Suite)
# ==============================================================================
import os, glob

run_dir = os.environ.get('LAST_RUN_DIR', None)
if not run_dir or not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")))
    if not candidates:
        raise FileNotFoundError(f"Không tìm thấy thư mục run cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")
eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest

if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Không tìm thấy checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Checkpoint được chọn để train downstream: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

# Huấn luyện toàn bộ probe Detection (Linear/MLP), Sizing (Ridge/MLP) và 5 model UDA (CORAL, DANN, DAN-MMD, CDAN, SHOT)
# Kết quả model weights sẽ được lưu tự động vào: <run_dir>/downstream_models/
!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.downstream.train_downstream --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --adaptation_mode all_test --device cuda --epochs 30 --uda_batch_size 128 --learning_rate 1e-3

[*] Checkpoint được chọn để train downstream: /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/checkpoints/best_cross_condition_jepa.pt
CANONICAL DOWNSTREAM & UDA TRAINING PIPELINE
  Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/checkpoints/best_cross_condition_jepa.pt
  Output Dir:       /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/downstream_models
  Adaptation Mode:  all_test -> ['cross_sensor', 'cross_waveform', 'cross_specimen', 'cross_liftoff']
  Device:           cuda

[STEP 1/3] Extracting Source Domain Pool (16 Base Domain files: Train + Val)...

  Source Detection samples: 60560 (Defects: 11600, Sound: 48960)
  Source Sizing samples:    7400 defect pixels

[STEP 2/3] Fitting & Saving Standard Probes (Linear/MLP Detection & Sizing)...
  Saved standard_probes.joblib to /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/downstream_models

[STEP 3/3] Trai

In [10]:
# ==============================================================================
# Full Downstream Evaluation + Readouts Diagnostic sau khi train xong
# ==============================================================================
import os, glob

run_dir = os.environ.get('LAST_RUN_DIR', None)
if not run_dir or not os.path.exists(run_dir):
    candidates = sorted(glob.glob(os.path.join(DRIVE_RUNS, f"{EXP_NAME}_*")))
    if not candidates:
        raise FileNotFoundError(f"Khong tim thay thu muc run nao cho {EXP_NAME} trong {DRIVE_RUNS}")
    run_dir = candidates[-1]

ckpt_best = os.path.join(run_dir, "checkpoints", "best_cross_condition_jepa.pt")
ckpt_latest = os.path.join(run_dir, "checkpoints", "latest_cross_condition_jepa.pt")

eval_ckpt = ckpt_best if os.path.exists(ckpt_best) else ckpt_latest
if not os.path.exists(eval_ckpt):
    raise FileNotFoundError(f"Khong tim thay checkpoint trong {os.path.join(run_dir, 'checkpoints')}")

print(f"[*] Dang evaluate checkpoint: {eval_ckpt}")
os.environ["EVAL_CKPT"] = eval_ckpt

!cd GraduationProject && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.evaluate --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda && KMP_DUPLICATE_LIB_OK=TRUE python -m src.PECT_JEPA.cross_condition.diagnose_readouts --checkpoint "$EVAL_CKPT" --data_dir "$DATA" --device cuda

[*] Dang evaluate checkpoint: /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/checkpoints/best_cross_condition_jepa.pt

PECT-JEPA Unified Downstream Benchmark Evaluation
Checkpoint:       /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/checkpoints/best_cross_condition_jepa.pt
Output Directory: /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/evaluation_results
Compute Device:   cuda
Target Splits:    ['train', 'val', 'test']

Discovered 81 total inspection files:
  - Train Split (Base Domain):     12 files
  - Val Split (Base Domain):       4 files
  - Test Split (Held-out OOD):     65 files

STEP 1: Downstream Probes & UDA Models Setup
  [INFO] Loading pre-trained downstream models from: /content/drive/MyDrive/PECT_Dataset/runs/exp66d_self_focus_20261020_182525/downstream_models
  Pre-trained standard downstream models loaded successfully!

STEP 2: Evaluating All Splits (Train, Val, Test)

>>> Running Eval